In [1]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from rdkit import Chem

from torch_geometric.data import Data

from torch_geometric.nn import (
    GCNConv,
    global_mean_pool
)

import faiss

In [2]:
df = pd.read_csv(
    "../data/processed/bindingdb_clean.csv"
)

df = df.sample(
    5000,
    random_state=42
).reset_index(drop=True)

print(df.shape)

(5000, 4)


smiles -> graph

In [3]:
def smiles_to_graph(
    smiles,
    target=0
):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        return None

    x = []

    for atom in mol.GetAtoms():

        x.append([
            atom.GetAtomicNum()
        ])

    edges = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()

        edges.append([i, j])
        edges.append([j, i])

    if len(edges) == 0:
        return None

    return Data(
        x=torch.tensor(
            x,
            dtype=torch.float
        ),

        edge_index=torch.tensor(
            edges,
            dtype=torch.long
        ).t()
    )

model class


In [4]:
class AffinityGNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.conv1 = GCNConv(1, 64)

        self.conv2 = GCNConv(64, 128)

        self.fc = nn.Linear(
            128,
            1
        )

    def encode(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.conv1(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = self.conv2(
            x,
            edge_index
        )

        x = torch.relu(x)

        x = global_mean_pool(
            x,
            batch
        )

        return x

    def forward(
        self,
        x,
        edge_index,
        batch
    ):

        x = self.encode(
            x,
            edge_index,
            batch
        )

        return self.fc(x)

Load checkPoint

In [5]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = AffinityGNN().to(device)

model.load_state_dict(
    torch.load(
        "../models/checkpoints/pharmagpt_v0_7.pt",
        map_location=device
    )
)

model.eval()

print("Model Loaded")

Model Loaded


Generate Embedding

In [6]:
embeddings = []
smiles_list = []

with torch.no_grad():

    for smiles in df["Ligand SMILES"]:

        graph = smiles_to_graph(
            smiles
        )

        if graph is None:
            continue

        graph = graph.to(device)

        batch = torch.zeros(
            graph.num_nodes,
            dtype=torch.long
        ).to(device)

        emb = model.encode(
            graph.x,
            graph.edge_index,
            batch
        )

        embeddings.append(
            emb.cpu().numpy()[0]
        )

        smiles_list.append(
            smiles
        )

print(
    "Embeddings:",
    len(embeddings)
)

print(
    "Dimension:",
    embeddings[0].shape
)

Embeddings: 4927
Dimension: (128,)


Build FAISS index

In [7]:
vectors = np.array(
    embeddings,
    dtype=np.float32
)

index = faiss.IndexFlatL2(
    vectors.shape[1]
)

index.add(vectors)

print(
    "Vectors:",
    index.ntotal
)

Vectors: 4927


similarity Search

In [8]:
query_id = 0

distances, indices = index.search(
    vectors[query_id].reshape(
        1,
        -1
    ),
    10
)

for rank, idx in enumerate(indices[0]):

    print(
        rank,
        distances[0][rank]
    )

    print(
        smiles_list[idx]
    )

    print()

0 0.0
CC(C)C1(CCc2ccc(O)cc2)CC(=O)C(Sc2cc(C)c(NS(=O)(=O)c3ccc(cc3)C#N)cc2C(C)(C)C)C(=O)O1

1 7.272049e-07
CC(C)C1(CCc2ccc(O)cc2)CC(=O)C(Sc2cc(C)c(NS(=O)(=O)c3cccc(c3)C#N)cc2C(C)(C)C)C(=O)O1

2 2.4277097e-06
CN(C)c1ccc(cc1)\N=N\c1ccc(cc1)S(=O)(=O)Nc1cccc(c1)B(O)O

3 2.8390355e-06
CC(C)CC(NC(C)=O)C(=O)N1CCC=C1P(O)(O)CC(Cc1ccccc1)C(O)=O |c:14|

4 7.028764e-06
CCCC1(C)SC(NCC2CCCCC2)=NC1=O |c:15|

5 8.179834e-06
COc1cccc(c1)S(=O)(=O)N(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)[C@@H]1CN(C(=O)O1)c1cccc(F)c1)C[C@H]1CCCO1 |r|

6 1.5925763e-05
O[C@H]([C@@H](O)[C@@H](Sc1ccccn1)C(=O)N[C@@H]1[C@H](O)Cc2ccccc12)[C@@H](Sc1ccccn1)C(=O)N[C@@H]1[C@H](O)Cc2ccccc12 |r|

7 1.9391133e-05
Cc1cc(C(=O)Nc2ccc(cc2)-c2ccccc2S(N)(=O)=O)n(n1)-c1ccc2onc(N)c2c1

8 2.0856332e-05
CCC(=S)NCc1c[nH]c2ccccc12

9 3.0251493e-05
Oc1cc(F)cc(F)c1

